# Assertions — Portfolio Construction, Timing, and Robustness

This notebook independently audits `notebooks/04_R_Portfolio.ipynb` and its published machine-readable outputs. It is a **non-mutating assertion barrier**: apart from one small deterministic equivalence test of the optimisation formulation, it does not regenerate a cache, overwrite an article artifact, or execute another notebook.

The checks cover the point-in-time holding engine, weight and cash accounting, transaction costs, performance reconstruction, expanding-window cross-validation, the three-level timing-permutation protocol, cross-universe results, robustness paths, and the final article artifact manifest.


## 1. Setup and audit policy

Paths resolve from either the repository root or `tests/`. Every assertion is recorded through `require`; execution stops at the first violated hard contract. Numerical comparisons use explicit tolerances and no displayed value is treated as evidence by itself.


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import cvxpy as cp

ROOT = Path.cwd()
if ROOT.name in {'notebooks', 'tests'}:
    ROOT = ROOT.parent

PORT = ROOT / 'outputs' / 'results' / 'portfolio'
AUDIT = PORT / 'audit'
IMAGES = PORT / 'images'
TABLES = PORT / 'tables'
NOTEBOOK = ROOT / 'notebooks' / '04_R_Portfolio.ipynb'
TIMING_CACHE_DIR = ROOT / 'cache' / 'portfolio'

CHECKS = []

def require(label, condition):
    ok = bool(condition)
    if not ok:
        raise AssertionError(label)
    CHECKS.append(label)
    print(f'PASS  {label}')

def file_signature(path):
    # Exact contract used by 04_R_Portfolio: SHA-256(path | size | mtime_ns).
    path = Path(path)
    stat = path.stat()
    payload = f'{path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}'
    return hashlib.sha256(payload.encode()).hexdigest()

def finite(frame, columns):
    return np.isfinite(frame[list(columns)].to_numpy(dtype=float)).all()

def annualised_sharpe(values):
    x = np.asarray(values, dtype=float)
    return np.sqrt(12.0) * x.mean() / x.std(ddof=1)

require('portfolio output directory exists', PORT.is_dir())
require('portfolio audit directory exists', AUDIT.is_dir())
require('article notebook exists', NOTEBOOK.is_file())


## 2. Signed engine artifacts and immutable identities

The engine manifest is the root contract. Its four canonical Parquet outputs must exist, match the recorded path--size--mtime signatures used by the production notebook, carry one common engine digest, and agree with the declared solver, window length, accounting convention, and sample coverage. These signatures are cache identities, not content hashes; numerical content is therefore validated independently in the following sections.


In [ ]:
ENGINE_FILES = [
    'portfolio_decisions.parquet',
    'portfolio_engine_audit.parquet',
    'portfolio_monthly.parquet',
    'portfolio_weight_ledger.parquet',
]
manifest_path = AUDIT / 'portfolio_engine_manifest.json'
require('engine manifest exists', manifest_path.is_file())
manifest = json.loads(manifest_path.read_text())

require('engine schema is direct-W2 PIT v2',
        manifest['engine'] == 'portfolio-pit-v2.0.0-direct-w2-socp')
require('engine uses a 36-month estimation window', manifest['W_EST'] == 36)
require('engine solver is MOSEK', manifest['solver'] == 'MOSEK')
require('initial state is 100 percent cash', manifest['initial_pretrade_state'] == '100pct_cash')
require('turnover excludes cash', manifest['turnover'] == 'sum_abs_risky_assets_cash_excluded')
require('net-return convention is explicit',
        manifest['net_return'] == 'gross_return_minus_cost_rate_times_turnover')

# Independent formulation guard. The production function must implement the
# closed-form W2 objective directly, while this test reconstructs the equivalent
# lambda dual with lambda*eps**2 and ||w||_2**2/(4*lambda).
production_nb = json.loads(NOTEBOOK.read_text())
w_dro_cells = [
    ''.join(cell.get('source', []))
    for cell in production_nb.get('cells', [])
    if 'def w_dro(R, eps):' in ''.join(cell.get('source', []))
]
require('production notebook defines exactly one w_dro function', len(w_dro_cells) == 1)
w_dro_source = w_dro_cells[0].split('def w_dro(R, eps):', 1)[1].split(
    '# --- raw daily returns pivot', 1
)[0]
require('production w_dro uses the direct W2 penalty',
        'float(eps) * cp.norm(w, 2)' in w_dro_source)
require('production w_dro no longer contains the legacy lambda dual',
        'quad_over_lin' not in w_dro_source and 'lam * eps' not in w_dro_source)

R_unit = np.array([
    [0.021,  0.006, -0.011,  0.014],
    [-0.009, 0.017,  0.004,  0.008],
    [0.013, -0.005,  0.019,  0.002],
    [0.004,  0.011, -0.003,  0.016],
    [0.018,  0.001,  0.007, -0.006],
], dtype=float)
eps_unit = 0.075
T_unit, N_unit = R_unit.shape

w_direct = cp.Variable(N_unit, nonneg=True)
direct_problem = cp.Problem(
    cp.Minimize(-R_unit.mean(axis=0) @ w_direct + eps_unit * cp.norm(w_direct, 2)),
    [cp.sum(w_direct) == 1],
)
direct_problem.solve(solver=cp.MOSEK)

w_dual = cp.Variable(N_unit, nonneg=True)
lam_dual = cp.Variable(nonneg=True)
s_dual = cp.Variable(T_unit)
dual_problem = cp.Problem(
    cp.Minimize(lam_dual * eps_unit**2 + cp.sum(s_dual) / T_unit),
    [
        cp.sum(w_dual) == 1,
        lam_dual >= 1e-8,
        *[
            s_dual[t] >= -R_unit[t] @ w_dual
            + cp.quad_over_lin(w_dual, 4 * lam_dual)
            for t in range(T_unit)
        ],
    ],
)
dual_problem.solve(solver=cp.MOSEK)

require('direct and dual W2 unit problems are optimal',
        direct_problem.status in {'optimal', 'optimal_inaccurate'}
        and dual_problem.status in {'optimal', 'optimal_inaccurate'})
np.testing.assert_allclose(w_direct.value, w_dual.value, rtol=0, atol=5e-5)
np.testing.assert_allclose(direct_problem.value, dual_problem.value, rtol=0, atol=2e-9)
require('direct SOCP equals the correctly parameterised W2 dual', True)

for name in ENGINE_FILES:
    path = AUDIT / name
    require(f'{name}: exists', path.is_file())
    require(f'{name}: non-empty', path.stat().st_size > 0)
    require(f'{name}: production file signature matches manifest',
            file_signature(path) == manifest['outputs'][name])

decisions = pd.read_parquet(AUDIT / 'portfolio_decisions.parquet')
engine_audit = pd.read_parquet(AUDIT / 'portfolio_engine_audit.parquet')
monthly = pd.read_parquet(AUDIT / 'portfolio_monthly.parquet')
ledger = pd.read_parquet(AUDIT / 'portfolio_weight_ledger.parquet')

for frame in (decisions, engine_audit, monthly, ledger):
    require('engine digest is unique within artifact',
            frame['engine_digest'].nunique() == 1)
    require('artifact engine digest matches manifest',
            frame['engine_digest'].iloc[0] == manifest['engine_digest'])


## 3. Decision calendar and holding-period accounting

There must be 373 formation decisions for each strategy but only 372 realised holding months: the final formation has no observable (M+1) return. Dates must be unique and consecutive, and every realised row must obey the exact gross, cost, net, and wealth recursions.


In [ ]:
STRATEGIES = {'static', 'dynamic'}
KEY = ['formation_month', 'strategy']
REAL_NUMERIC = [
    'epsilon', 'rho', 'turnover', 'cost_bps', 'transaction_cost',
    'gross_factor', 'gross_return', 'net_return',
    'pre_risky_sum', 'pre_cash_weight', 'target_weight_sum',
    'end_risky_sum', 'end_cash_weight', 'gross_reconciliation_error',
]

require('decision strategies are exactly static and dynamic',
        set(decisions['strategy']) == STRATEGIES)
require('decision key is unique', not decisions.duplicated(KEY).any())
require('monthly key is unique', not monthly.duplicated(KEY).any())
require('engine-audit key is unique', not engine_audit.duplicated(KEY).any())

decision_counts = decisions.groupby('strategy').size()
realised_counts = decisions.loc[decisions['realised']].groupby('strategy').size()
require('decision count matches manifest',
        decision_counts.eq(manifest['decisions_per_strategy']).all())
require('realised count matches manifest',
        realised_counts.eq(manifest['realised_months_per_strategy']).all())
require('one terminal unrealised decision per strategy',
        decisions.loc[~decisions['realised']].groupby('strategy').size().eq(1).all())
require('monthly output contains realised rows only', monthly['realised'].all())

for strategy, group in monthly.groupby('strategy'):
    group = group.sort_values('return_date')
    periods = pd.PeriodIndex(pd.to_datetime(group['return_date']), freq='M')
    expected = pd.period_range(manifest['first_return'], manifest['last_return'], freq='M')
    require(f'{strategy}: complete consecutive return calendar',
            periods.equals(expected))

realised = decisions.loc[decisions['realised']].copy()
require('realised numerical fields are finite', finite(realised, REAL_NUMERIC))
require('rho is non-negative', (realised['rho'] >= 0).all())
require('epsilon respects engine bounds',
        realised['epsilon'].between(manifest['eps_floor'], manifest['eps_cap']).all())
require('formation precedes holding month',
        (pd.to_datetime(realised['formation_month']).dt.to_period('M') + 1
         == pd.to_datetime(realised['holding_month']).dt.to_period('M')).all())
require('return date belongs to holding month',
        (pd.to_datetime(realised['return_date']).dt.to_period('M')
         == pd.to_datetime(realised['holding_month']).dt.to_period('M')).all())

np.testing.assert_allclose(realised['transaction_cost'],
                           realised['cost_bps'] / 1e4 * realised['turnover'],
                           rtol=0, atol=1e-14)
require('transaction cost identity', True)
np.testing.assert_allclose(realised['gross_factor'], 1.0 + realised['gross_return'],
                           rtol=0, atol=1e-14)
require('gross-factor identity', True)
np.testing.assert_allclose(realised['net_return'],
                           realised['gross_return'] - realised['transaction_cost'],
                           rtol=0, atol=1e-14)
require('net-return identity', True)
require('gross reconciliation error is negligible',
        realised['gross_reconciliation_error'].abs().max() < 1e-12)

for strategy, group in monthly.groupby('strategy'):
    group = group.sort_values('return_date')
    gross_wealth = (1.0 + group['gross_return']).cumprod()
    net_wealth = (1.0 + group['net_return']).cumprod()
    np.testing.assert_allclose(group['gross_wealth'], gross_wealth, rtol=0, atol=2e-13)
    np.testing.assert_allclose(group['net_wealth'], net_wealth, rtol=0, atol=2e-13)
require('gross and net wealth paths reconstruct exactly', True)


## 4. Weight ledger, cash, delisting, and turnover reconciliation

The ledger is the security-level proof of the engine. Pre-trade and target states must each sum to one, weights must be long-only, cash must never enter turnover, and realised post-holding states must sum to one. A delisted security may disappear into cash during (M+1), but its terminal proceeds remain inside the self-financing state until the next rebalance.


In [ ]:
LEDGER_KEY = ['formation_month', 'strategy', 'PERMNO']
require('ledger key is unique', not ledger.duplicated(LEDGER_KEY).any())
require('exactly one cash row per decision',
        ledger.groupby(KEY)['is_cash'].sum().eq(1).all())
require('cash contributes zero to reported turnover',
        ledger.loc[ledger['is_cash'], 'turnover_component'].abs().max() < 1e-15)
require('weights and turnover components are non-negative',
        (ledger[['pre_weight', 'target_weight', 'turnover_component']]
         .fillna(0) >= -1e-14).all().all())

ledger_group = (
    ledger.groupby(KEY, as_index=False)
    .agg(
        pre_sum=('pre_weight', 'sum'),
        target_sum=('target_weight', 'sum'),
        turnover_rebuilt=('turnover_component', 'sum'),
        end_sum=('end_weight', 'sum'),
    )
)
joined = engine_audit.merge(
    ledger_group, on=KEY, how='left', validate='one_to_one',
    suffixes=('_stored', '_ledger')
)
require('every engine decision has one ledger aggregate',
        len(joined) == len(engine_audit) and joined['pre_sum_ledger'].notna().all())
for stem in ['pre_sum', 'target_sum', 'turnover_rebuilt']:
    np.testing.assert_allclose(
        joined[f'{stem}_stored'], joined[f'{stem}_ledger'],
        rtol=0, atol=2e-13
    )
require('ledger reproduces pre, target, and turnover audit totals', True)
require('pre-trade states sum to one',
        np.max(np.abs(joined['pre_sum_ledger'] - 1.0)) < 2e-13)
require('target states sum to one',
        np.max(np.abs(joined['target_sum_ledger'] - 1.0)) < 2e-13)
require('realised end states sum to one',
        np.max(np.abs(joined.loc[joined['realised'], 'end_sum_ledger'] - 1.0)) < 2e-13)
np.testing.assert_allclose(
    joined['turnover'], joined['turnover_rebuilt_ledger'],
    rtol=0, atol=2e-13
)
require('reported turnover equals risky-asset ledger turnover', True)

delisted = realised.loc[realised['n_delisted'] > 0]
require('delisting events, if present, preserve non-negative cash',
        delisted.empty or (delisted['end_cash_weight'] >= -1e-14).all())
print(f'documented realised delisting months: {len(delisted)}')


## 5. Independent performance reconstruction

The principal static/dynamic table is rebuilt from monthly returns rather than trusted as a frozen display. Annual return, CAGR, volatility, Sharpe, and turnover are checked independently; the reported (Delta)Sharpe must equal dynamic minus static for both gross and net bases. Table turnover excludes the one-off initial deployment from cash, while the engine-level monthly series retains it.


In [ ]:
metrics = pd.read_parquet(AUDIT / 'P_01_static_dynamic_metrics.parquet')
inference = pd.read_parquet(AUDIT / 'P_01_static_dynamic_inference.parquet')
require('performance metric key is unique',
        not metrics.duplicated(['strategy', 'basis']).any())
require('performance table covers both strategies and bases',
        set(map(tuple, metrics[['strategy', 'basis']].to_records(index=False)))
        == {(s, b) for s in STRATEGIES for b in {'gross', 'net'}})

for _, row in metrics.iterrows():
    group = monthly.loc[monthly['strategy'].eq(row['strategy'])].sort_values('return_date')
    returns = group[f"{row['basis']}_return"].to_numpy()
    expected = {
        'annual_return': 12.0 * returns.mean(),
        'cagr': np.prod(1.0 + returns) ** (12.0 / len(returns)) - 1.0,
        'volatility': np.sqrt(12.0) * returns.std(ddof=1),
        'Sharpe': annualised_sharpe(returns),
        'turnover': group['turnover'].iloc[1:].mean(),
    }
    for field, value in expected.items():
        np.testing.assert_allclose(row[field], value, rtol=0, atol=2e-12)
require('principal performance metrics reconstruct from monthly paths', True)

for basis in ['gross', 'net']:
    table = metrics.loc[metrics['basis'].eq(basis)].set_index('strategy')
    expected_delta = table.loc['dynamic', 'Sharpe'] - table.loc['static', 'Sharpe']
    stored = inference.loc[inference['basis'].eq(basis), 'delta_sharpe'].item()
    np.testing.assert_allclose(stored, expected_delta, rtol=0, atol=2e-12)
require('gross and net delta-Sharpe identities', True)


## 6. Expanding-window cross-validation without look-ahead

The CV path begins in January 1995 with a predeclared, potentially off-grid radius for 36 holding months. From month 37 onward, each selected radius must be the maximiser of the Sharpe computed from **strictly prior** realised candidate returns. Every month must contain the complete 16-point grid, and each post-initialisation selected candidate return must reproduce the published CV portfolio before transaction costs.


In [ ]:
cv_candidates = pd.read_parquet(AUDIT / 'P_05_cv_candidate_monthly.parquet')
cv_scores = pd.read_parquet(AUDIT / 'P_05_cv_expanding_scores.parquet')
cv_select = pd.read_parquet(AUDIT / 'P_05_cv_selection_path.parquet')
cv_monthly = pd.read_parquet(AUDIT / 'P_05_cv_portfolio_monthly.parquet')

cv_digest = cv_candidates['cv_digest'].iloc[0]
for name, frame in {
    'candidate': cv_candidates, 'score': cv_scores,
    'selection': cv_select, 'monthly': cv_monthly,
}.items():
    require(f'CV {name}: one common digest',
            frame['cv_digest'].nunique() == 1 and frame['cv_digest'].iloc[0] == cv_digest)

require('CV candidate key is unique',
        not cv_candidates.duplicated(['formation_month', 'epsilon']).any())
require('CV grid has exactly 16 radii every month',
        cv_candidates.groupby('formation_month')['epsilon'].nunique().eq(16).all())
require('CV candidate panel has 372 months',
        cv_candidates['formation_month'].nunique() == 372)
require('CV selection has one row per formation month',
        len(cv_select) == 372 and not cv_select.duplicated('formation_month').any())
require('CV initialisation is exactly 36 months',
        cv_select['is_initialisation'].sum() == 36)
require('initialisation source is explicit',
        cv_select.loc[cv_select['is_initialisation'], 'selection_source']
        .eq('predeclared_initialisation').all())
require('post-initialisation uses expanding Sharpe',
        cv_select.loc[~cv_select['is_initialisation'], 'selection_source']
        .eq('expanding_max_sharpe').all())

post = cv_select.loc[~cv_select['is_initialisation']].copy()
require('first estimated CV choice uses 36 prior observations',
        post['prior_observations'].min() == 36)
require('expanding prior count reaches 371',
        post['prior_observations'].max() == 371)
require('CV score grid is complete after initialisation',
        cv_scores.groupby('formation_month')['epsilon'].nunique().eq(16).all())

winners = (
    cv_scores.sort_values(
        ['formation_month', 'prior_sharpe', 'epsilon'],
        ascending=[True, False, True]
    )
    .groupby('formation_month', as_index=False)
    .first()[['formation_month', 'epsilon', 'prior_sharpe', 'prior_observations']]
)
selected_check = post.merge(winners, on='formation_month', validate='one_to_one')
np.testing.assert_allclose(selected_check['selected_epsilon'],
                           selected_check['epsilon'], rtol=0, atol=1e-12)
np.testing.assert_allclose(selected_check['selected_prior_sharpe'],
                           selected_check['prior_sharpe'], rtol=0, atol=1e-12)
require('every post-initialisation radius is the prior-Sharpe maximiser', True)

# The 36-month initial radius is deliberately predeclared and off-grid; only
# post-initialisation choices have a candidate-grid return to reconcile.
require('initial radius is constant and finite',
        cv_select.loc[cv_select['is_initialisation'], 'selected_epsilon'].nunique() == 1
        and np.isfinite(cv_select.loc[cv_select['is_initialisation'], 'selected_epsilon']).all())
picked = post[['formation_month', 'selected_epsilon']].merge(
    cv_candidates,
    left_on=['formation_month', 'selected_epsilon'],
    right_on=['formation_month', 'epsilon'],
    how='left', validate='one_to_one'
)
cv_check = cv_monthly.loc[
    cv_monthly['formation_month'].isin(post['formation_month'])
].merge(
    picked[['formation_month', 'gross_return']],
    on='formation_month', validate='one_to_one',
    suffixes=('_portfolio', '_candidate')
)
np.testing.assert_allclose(cv_check['gross_return_portfolio'],
                           cv_check['gross_return_candidate'], rtol=0, atol=2e-12)
np.testing.assert_allclose(cv_monthly['net_return'],
                           cv_monthly['gross_return'] - cv_monthly['transaction_cost'],
                           rtol=0, atol=2e-14)
require('selected CV candidate and net accounting reconstruct', True)


## 7. Three-level timing-permutation inference

This is the authoritative replacement for the earlier independent summaries.

1. Level 1 tests the large-cap dynamic Sharpe under monthly, 6-month-block, and 12-month-block nulls.
2. Level 2 uses one shared monthly permutation per draw and controls the four reported cells ((L_{
m all},L_{
m exdc},S_{
m all},S_{
m exdc})).
3. Level 3 uses the same simple-permutation draws but expands the family to 12 predeclared cells.

The audit reconstructs every exceedance count and finite-sample (p=(r+1)/(B+1)), and independently verifies both row-wise max-T statistics.


In [ ]:
timing_manifests = sorted(TIMING_CACHE_DIR.glob('timing_test_manifest_*.json'))
require('exactly one signed timing manifest is active', len(timing_manifests) == 1)
TIMING_MANIFEST = timing_manifests[0]
timing_manifest = json.loads(TIMING_MANIFEST.read_text())
digest = timing_manifest['experiment_digest']
B = timing_manifest['draws']
cells4 = timing_manifest['cells4']
cells12 = timing_manifest['cells12']

require('timing schema is direct-W2 three-level v2',
        timing_manifest['schema_version'] == 'portfolio-timing-three-level-v2.0.0')
require('timing solver contract uses the direct W2 SOCP',
        timing_manifest['solver_contract']['portfolio_engine']
        == 'portfolio-pit-v2.0.0-direct-w2-socp'
        and timing_manifest['solver_contract']['formulation']
        == 'linear-loss-direct-W2-SOCP')
require('timing primary basis is net', timing_manifest['primary_basis'] == 'net')
require('timing target is S_exdc', timing_manifest['target_cell'] == 'S_exdc')
require('timing experiment has 1000 draws', B == 1000)
require('simple permutation is shared across L and S',
        'shared by L and S' in timing_manifest['dependence_contract']['simple'])
require('predeclared families have 4 and 12 cells',
        len(cells4) == 4 and len(cells12) == 12 and set(cells4).issubset(cells12))

draws = pd.read_parquet(AUDIT / 'P_06_timing_tests_draws.parquet').sort_values('draw')
placebo = pd.read_parquet(
    AUDIT / 'P_02_placebo_inference.parquet'
)
maxt4 = pd.read_parquet(AUDIT / 'P_07_maxt4_inference.parquet')
maxt12 = pd.read_parquet(AUDIT / 'P_08_maxt12_inference.parquet')

require('draw indices are exactly 0 through B-1',
        np.array_equal(draws['draw'].to_numpy(), np.arange(B)))
require('each timing seed stream is unique',
        all(draws[c].nunique() == B for c in ['seed_simple', 'seed_block6', 'seed_block12']))
require('draw artifact has one experiment digest',
        draws['experiment_digest'].nunique() == 1
        and draws['experiment_digest'].iloc[0] == digest)
require('inference artifacts carry the experiment digest',
        all(f['experiment_digest'].nunique() == 1
            and f['experiment_digest'].iloc[0] == digest
            for f in [placebo, maxt4, maxt12]))

for basis in ['net', 'gross']:
    np.testing.assert_allclose(
        draws[f'{basis}_maxT4'],
        draws[[f'{basis}_{cell}' for cell in cells4]].max(axis=1),
        rtol=0, atol=1e-15
    )
    np.testing.assert_allclose(
        draws[f'{basis}_maxT12'],
        draws[[f'{basis}_{cell}' for cell in cells12]].max(axis=1),
        rtol=0, atol=1e-15
    )
require('maxT4 and maxT12 are exact row-wise family maxima', True)

for _, row in placebo.iterrows():
    col = f"L_{row['method']}_sharpe_{row['basis']}"
    r = int((draws[col] >= row['observed_sharpe']).sum())
    require(f"level 1 {row['basis']} {row['method']}: exceedance count",
            r == row['exceedances'])
    np.testing.assert_allclose(row['p_value'], (r + 1) / (B + 1), rtol=0, atol=1e-15)
    np.testing.assert_allclose(row['null_mean'], draws[col].mean(), rtol=0, atol=1e-15)
    np.testing.assert_allclose(row['null_q95'], draws[col].quantile(.95), rtol=0, atol=1e-15)
require('all level-1 placebo p-values reconstruct', True)

def verify_family(frame, expected_cells):
    require('family artifact has exact cells and bases',
            set(frame['cell']) == set(expected_cells)
            and set(frame['basis']) == {'net', 'gross'})
    require('family cell-basis key is unique',
            not frame.duplicated(['cell', 'basis']).any())
    for _, row in frame.iterrows():
        basis, cell = row['basis'], row['cell']
        observed = row['observed_delta_sharpe']
        r_marginal = int((draws[f'{basis}_{cell}'] >= observed).sum())
        r4 = int((draws[f'{basis}_maxT4'] >= observed).sum())
        r12 = int((draws[f'{basis}_maxT12'] >= observed).sum())
        assert r_marginal == row['marginal_exceedances']
        if cell in cells4:
            assert r4 == row['maxT4_exceedances']
            np.testing.assert_allclose(row['p_maxT4'], (r4 + 1)/(B + 1), atol=1e-15)
        else:
            # Level-3-only cells are not members of the four-cell family.
            assert pd.isna(row['maxT4_exceedances']) and pd.isna(row['p_maxT4'])
        assert r12 == row['maxT12_exceedances']
        np.testing.assert_allclose(row['p_marginal'], (r_marginal + 1)/(B + 1), atol=1e-15)
        np.testing.assert_allclose(row['p_maxT12'], (r12 + 1)/(B + 1), atol=1e-15)

verify_family(maxt4, cells4)
verify_family(maxt12, cells12)
require('all marginal, maxT4, and maxT12 p-values reconstruct', True)

target_rows = maxt4.query("basis == 'net' and cell == 'S_exdc'")
require('authoritative target row is unique', len(target_rows) == 1)
target = target_rows.iloc[0]
resolution = 1.0 / (B + 1)
require('target p-values lie on the finite-sample permutation grid',
        all(
            resolution <= float(target[column]) <= 1.0
            and np.isclose(float(target[column]) / resolution,
                           round(float(target[column]) / resolution), atol=1e-12)
            for column in ['p_marginal', 'p_maxT4', 'p_maxT12']
        ))
print('\nAuthoritative net inference:')
print(placebo.query("basis == 'net'")[['method', 'observed_sharpe', 'p_value']].to_string(index=False))
print(f"S_exdc: p_marginal={target['p_marginal']:.4f}, "
      f"p_maxT4={target['p_maxT4']:.4f}, p_maxT12={target['p_maxT12']:.4f}")


## 8. Cross-universe and temporal-stability reconstruction

The large-cap and NYSE small–mid-cap paths must cover the identical 372 holding months for static and dynamic strategies. Full-sample and dot-com-excluded Sharpe gains are recomputed from returns, and the three disjoint decades must contain 120, 120, and 132 months respectively.


In [ ]:
universe_monthly = pd.read_parquet(AUDIT / 'P_06_universe_monthly.parquet')
universe_observed = pd.read_parquet(AUDIT / 'P_06_universe_observed.parquet')
temporal = pd.read_parquet(AUDIT / 'P_09_temporal_stability.parquet')

UKEY = ['universe', 'formation_month', 'strategy']
require('universe monthly key is unique',
        not universe_monthly.duplicated(UKEY).any())
require('two universes and two strategies are complete',
        set(universe_monthly['universe']) == {'big_caps', 'small_caps_p20_p50'}
        and set(universe_monthly['strategy']) == STRATEGIES)
require('every universe-strategy path has 372 months',
        universe_monthly.groupby(['universe', 'strategy']).size().eq(372).all())
require('universe paths share one return calendar',
        universe_monthly.groupby(['universe', 'strategy'])['return_date']
        .apply(lambda x: tuple(pd.to_datetime(x).sort_values()))
        .nunique() == 1)
np.testing.assert_allclose(
    universe_monthly['net_return'],
    universe_monthly['gross_return'] - universe_monthly['transaction_cost'],
    rtol=0, atol=2e-14
)
require('universe net-return accounting reconstructs', True)

for _, row in universe_observed.iterrows():
    sample = universe_monthly.loc[
        universe_monthly['universe'].eq(row['universe'])
    ].copy()
    if row['sample'] == 'ex_dotcom':
        period = pd.to_datetime(sample['return_date']).dt.to_period('M')
        sample = sample.loc[~period.between(pd.Period('1999-01'), pd.Period('2001-12'))]
    elif row['sample'] != 'full':
        raise AssertionError(f"unknown sample {row['sample']}")
    static = sample.loc[sample['strategy'].eq('static'), f"{row['basis']}_return"]
    dynamic = sample.loc[sample['strategy'].eq('dynamic'), f"{row['basis']}_return"]
    static_sh = annualised_sharpe(static)
    dynamic_sh = annualised_sharpe(dynamic)
    np.testing.assert_allclose(row['static_sharpe'], static_sh, rtol=0, atol=2e-12)
    np.testing.assert_allclose(row['dynamic_sharpe'], dynamic_sh, rtol=0, atol=2e-12)
    np.testing.assert_allclose(row['delta_sharpe'], dynamic_sh-static_sh, rtol=0, atol=2e-12)
    for strategy, field in [('static', 'static_turnover'), ('dynamic', 'dynamic_turnover')]:
        turnover = (sample.loc[sample['strategy'].eq(strategy)]
                    .sort_values('return_date')['turnover'].reset_index(drop=True))
        assert np.isclose(turnover.iloc[0], 1.0, rtol=0.0, atol=1e-12)
        np.testing.assert_allclose(row[field], turnover.iloc[1:].mean(), rtol=0, atol=2e-12)
    assert row['n_months'] == len(static)
require('all full and ex-dot-com universe Sharpe gains reconstruct', True)
require('universe summaries exclude initial deployment from recurring turnover', True)

require('temporal table has two universes, three periods, and two bases',
        len(temporal) == 12)
expected_n = {'1995--2004': 120, '2005--2014': 120, '2015--2025': 132}
require('disjoint temporal windows have declared month counts',
        temporal.apply(lambda r: r['n_months'] == expected_n[r['period']], axis=1).all())
np.testing.assert_allclose(
    temporal['delta_sharpe'],
    temporal['dynamic_sharpe'] - temporal['static_sharpe'],
    rtol=0, atol=2e-12
)
require('temporal delta-Sharpe identity', True)


## 9. Robustness caches and article artifact manifest

Robustness results are accepted only when each solver status is optimal, weights sum to one, monthly keys are unique, and numerical outputs are finite. The final publication check requires all 14 figures and all declared LaTeX tables to exist and be non-empty. It also rejects saved error tracebacks in the article notebook.


In [ ]:
rob_metrics = pd.read_parquet(AUDIT / 'P_09_robustness_metrics.parquet')
rob_monthly = pd.read_parquet(AUDIT / 'P_09_robustness_monthly.parquet')
rob_solver = pd.read_parquet(AUDIT / 'P_09_robustness_solver_audit.parquet')
useful_solver = pd.read_parquet(AUDIT / 'P_09_useful_radius_solver_audit.parquet')
useful_summary = pd.read_parquet(AUDIT / 'P_09_useful_radius_summary.parquet')

for name, frame in {'robustness': rob_solver, 'useful-radius': useful_solver}.items():
    require(f'{name}: all solver statuses are optimal',
            frame['solver_status'].astype(str).str.lower().str.startswith('optimal').all())
    require(f'{name}: weights sum to one',
            np.max(np.abs(frame['weight_sum'] - 1.0)) < 2e-10)
    require(f'{name}: finite radii and concentrations',
            finite(frame, ['epsilon', 'weight_sum', 'weight_l2']))

require('robustness metric key is unique',
        not rob_metrics.duplicated(['lever', 'value']).any())
require('robustness monthly path key is unique',
        not rob_monthly.duplicated(['path_id', 'formation_month']).any())
require('robustness return and cost fields are finite',
        finite(rob_monthly, ['gross_return', 'net_return', 'turnover', 'transaction_cost']))
np.testing.assert_allclose(
    rob_monthly['net_return'],
    rob_monthly['gross_return'] - rob_monthly['transaction_cost'],
    rtol=0, atol=2e-14
)
require('robustness net-return identity', True)
require('useful-radius grid is strictly increasing',
        np.all(np.diff(useful_summary['base_radius'].to_numpy()) > 0))
require('reference radius is present in useful-radius grid',
        np.isclose(useful_summary['base_radius'], 0.357661, atol=5e-7).any())

EXPECTED_FIGURES = [f'R_P_{i:02d}_{name}.pdf' for i, name in [
    (1, 'radius_time'), (2, 'concentration_vs_eps'), (3, 'relative_gain'),
    (4, 'placebo_timing'), (5, 'conditional_attribution'), (6, 'landscape'),
    (7, 'degeneracy'), (8, 'universe_gain'), (9, 'maxt_multiplicity'),
    (10, 'useful_window'), (11, 'robustness_levers'), (12, 'decades_universe'),
    (13, 'radius_forms_fullsample'), (14, 'crises_signal_forms'),
]]
EXPECTED_TABLES = [
    'P_01_static_dynamic_metrics.tex', 'P_02_placebo.tex',
    'P_03_conditional_attribution.tex', 'P_04_landscape.tex',
    'P_05_degeneracy.tex', 'P_06_universe.tex',
    'P_07_maxt4.tex', 'P_08_maxt12.tex',
    'P_07_robustness_metrics.tex', 'P_09_robustness_metrics.tex',
    'P_10_volatility_benchmark.tex',
]
for name in EXPECTED_FIGURES:
    path = IMAGES / name
    require(f'figure {name}: exists and is non-empty',
            path.is_file() and path.stat().st_size > 0)
for name in EXPECTED_TABLES:
    path = TABLES / name
    require(f'table {name}: exists and is non-empty',
            path.is_file() and path.stat().st_size > 0)

# The article-facing performance tables intentionally retain only the parsimonious
# metric set; CAGR, Sortino, and Calmar remain available in the Parquet audit.
for name in ['P_01_static_dynamic_metrics.tex', 'P_04_landscape.tex']:
    table_text = (TABLES / name).read_text()
    for omitted_metric in ['CAGR', 'Sortino', 'Calmar']:
        require(f'{name}: excludes {omitted_metric} from article presentation',
                omitted_metric not in table_text)

article_nb = json.loads(NOTEBOOK.read_text())
presentation_markers = ['# FIG P_03', '# FIG P_06']
presentation_cells = [
    ''.join(cell.get('source', []))
    for cell in article_nb.get('cells', [])
    if any(marker in ''.join(cell.get('source', [])) for marker in presentation_markers)
]
require('article presentation cells are uniquely identified',
        len(presentation_cells) == len(presentation_markers))
presentation_source = '\n'.join(presentation_cells)
for omitted_metric in ['CAGR', 'Sortino', 'Calmar']:
    require(f'article presentation source excludes {omitted_metric}',
            omitted_metric not in presentation_source)

saved_errors = [
    output
    for cell in article_nb.get('cells', [])
    for output in cell.get('outputs', [])
    if output.get('output_type') == 'error'
]
require('article notebook contains no saved error outputs', len(saved_errors) == 0)


## 10. Conclusion

Reaching this cell means that the portfolio engine, ledger accounting, principal metrics, expanding CV, three-level timing inference, cross-universe paths, robustness caches, and article artifacts satisfy their hard reproducibility contracts.

A passing suite establishes **internal correctness and reproducibility**. It does not turn descriptive economic results into causal evidence, nor does it replace substantive interpretation of the weak cells and the loss of significance under the 12-cell family.


In [ ]:
require('portfolio audit reached final cell', True)
print('\n' + '=' * 76)
print(f'ALL HARD CONTRACTS PASSED — {len(CHECKS)} checks')
print('Portfolio validation is internally reproducible and ready for article use.')
print('=' * 76)